# Práctico: Introducción a Modelos de Lenguaje usando Hugging Face y GPT-2

In [1]:
# Primero, instala la librería transformers si no está instalada
# Descomenta la línea de abajo si necesitas instalar la librería
# !pip install transformers

In [2]:
#%pip install transformers

In [3]:
# Importando las librerías necesarias
import time
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import logging as transformers_logging

# Configurar el logging para que no muestre warnings de transformers
transformers_logging.set_verbosity_error()

## 1. Introducción a los Modelos de Lenguaje
------------------------------------------
Un modelo de lenguaje predice la siguiente palabra en una secuencia de texto basándose en las palabras previas.

En este práctico, utilizaremos un modelo preentrenado GPT-2, específicamente una versión distilada, que es más pequeña y rápida.

## 2. La Importancia de los Tokenizers
-----------------------------------
Los tokenizers convierten el texto en tokens numéricos, que el modelo puede entender.
Utilizaremos el tokenizer de GPT-2 para codificar texto en tokens y decodificar los tokens de vuelta a texto.

In [4]:
# Cargando el tokenizer
tokenizer = AutoTokenizer.from_pretrained('distilgpt2')

In [5]:
# Ejemplo: Tokenizando una oración
sentence = "A long time ago in a galaxy far, far away there was an astrophysicist"

In [6]:
# Tokenizar la oración
tokens = tokenizer.encode(sentence)
print("Oración tokenizada:", tokens)

Oración tokenizada: [32, 890, 640, 2084, 287, 257, 16161, 1290, 11, 1290, 1497, 612, 373, 281, 48782, 893, 48187]


Los tokens son los números que representan las palabras o partes de las palabras.

In [7]:
# Decodificando los tokens de nuevo a texto
decoded_sentence = tokenizer.decode(tokens)
print("Oración decodificada:", decoded_sentence)

Oración decodificada: A long time ago in a galaxy far, far away there was an astrophysicist


### Tokenizador de GPT-2 vs Tokenizador BERT.

In [8]:
# Cargar el tokenizer de BERT
bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

print("BERT tokenizer:")

# Tokenizar con BERT
bert_tokens = bert_tokenizer.encode(sentence)
print("- Tokens:", bert_tokens)
print("- Cantidad de tokens:", len(bert_tokens))

# Tokens individuales
indivitual_tokens = bert_tokenizer.convert_ids_to_tokens(bert_tokens)
print("- Tokens individuales: ", indivitual_tokens)

# Decodificar los tokens de BERT
bert_decoded_sentence = ...
print("- Oración decodificada:", bert_decoded_sentence, '\n')


print("GPT-2 tokenizer:")
# Comparar con el tokenizer de GPT-2
gpt2_tokens = tokenizer.encode(sentence)
print("- Tokens:", gpt2_tokens)
print("- Cantidad de tokens:", len(gpt2_tokens))

# Tokens individuales
gpt2_indivitual_tokens = ...
print("- Tokens individuales: ", gpt2_indivitual_tokens)

# Decodificar los tokens de GPT-2
gpt2_decoded_sentence = ...
print("- Oración decodificada:", gpt2_decoded_sentence)


BERT tokenizer:
- Tokens: [101, 1037, 2146, 2051, 3283, 1999, 1037, 9088, 2521, 1010, 2521, 2185, 2045, 2001, 2019, 28625, 21281, 19570, 2923, 102]
- Cantidad de tokens: 20
- Tokens individuales:  ['[CLS]', 'a', 'long', 'time', 'ago', 'in', 'a', 'galaxy', 'far', ',', 'far', 'away', 'there', 'was', 'an', 'astro', '##phy', '##sic', '##ist', '[SEP]']
- Oración decodificada: Ellipsis 

GPT-2 tokenizer:
- Tokens: [32, 890, 640, 2084, 287, 257, 16161, 1290, 11, 1290, 1497, 612, 373, 281, 48782, 893, 48187]
- Cantidad de tokens: 17
- Tokens individuales:  Ellipsis
- Oración decodificada: Ellipsis


### Ejercicio:
- ¿Qué son los tokens `[CLS]` y `[SEP]` que aparecen en la *Oracion decodificada* de BERT?

Estos son tokens que usa BERT para definir el inicio y fin de una secuencia respectivamente. En el caso de [CLS] se usa para la representación de toda la oración en tareas de clasificación. [SEP] no solo marca el final de una oración, sino que además sirve para separar oraciones.

- ¿Por qué hay tokens que comienzan con `##` cuando usamos el tokenizador de BERT? ¿Qué significan? ¿Y los que no lo tienen?

Esto se debe a que BERT usa WordPiece, por lo que "##" indica que el token es continuación de la palabra anterior, sin espacios. Un ejemplo de esto sería ver "astro" "##physic" "ist" que en teoría formaría una sola palabra. Esta es la forma que tiene BERT de manejar palabras complejas sin necesidad de tener un vocabulario enorme.

- ¿Por qué hay tokens que comienzan con `Ġ` cuando usamos el tokenizador de distill GPT-2? ¿Qué significan? ¿Y los que no lo tienen?

En GPT-2 se usa BPE a nivel bytes y "Ġ" representa un espacio en blanco (byte 0x20), el cual se encuentra previo a un token. Esto nos ayuda a entender cuando un token es el comienzo de una palabra, ya que este se ubica luego de un espacio. Aquellos casos que no tengan este término serán continuaciones de una oración. Esto es contrario a BERT, que marca continuaciones y aquí estamos marcando inicios.

- ¿Cree que son necesarios estos tokens? ¿Por qué?

Estos son importantes para BERT ya que el modelo fué entrenado con ellos y este espera esa estructura. Sin estos tokens, el modelo tiende a bajar su rendimiento en casos que estemos haciendo tareas de clasificación o con al menos 2 oraciónes.

- ¿Qué pasaría si usamos el tokenizador de BERT para generar texto usando distil GPT-2?

Esto no funcionaría ya que cada modelo tiene su propio vocabulario y matriz de embeddings, donde el mismo ID puede significar dos tokens totalmente distintos en cada modelo de lenguaje, con tamaños de lenguaje diferentes y con maneras distintas de separar tokens. 

## 3. Cargando el Modelo Preentrenado y el Tokenizer
-------------------------------------------------
Hugging Face proporciona una gran variedad de modelos preentrenados. Usaremos el modelo 'distilgpt2' para este ejercicio.

In [9]:
# Cargar el modelo
model = AutoModelForCausalLM.from_pretrained('distilgpt2')

W1031 15:08:38.243000 95692 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

In [10]:
# Poner el modelo en modo de evaluación
model.eval()

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

## 4. Salida Cruda (Logits)
--------------------------------------------
Obtendremos la salida cruda del modelo (logits).

Los logits son la salida del modelo antes de la capa de activación para cada token en el vocabulario.

In [11]:
# Ejemplo de prompt
prompt = "Yesterday, I dreamed of a world where"

In [12]:
# Tokenizar la entrada
inputs = tokenizer(prompt, return_tensors="pt")

In [13]:
inputs["input_ids"].shape

torch.Size([1, 8])

In [14]:
# Pasar la entrada por el modelo para obtener los logits (puntajes sin normalizar)
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits  # Salida cruda del modelo

In [15]:
logits

tensor([[[-34.2139, -32.5890, -34.5217,  ..., -45.8649, -45.4755, -34.5369],
         [-73.7792, -73.7084, -73.1784,  ..., -77.2609, -76.5639, -71.1657],
         [-80.8371, -80.4167, -82.7387,  ..., -88.8970, -81.7753, -82.5226],
         ...,
         [-73.3640, -73.6931, -76.6877,  ..., -82.1660, -78.2536, -73.7447],
         [-56.6977, -59.0597, -63.5706,  ..., -69.0110, -65.6015, -60.3717],
         [-59.4074, -60.2300, -63.9870,  ..., -63.8053, -65.4991, -62.3250]]])

In [16]:
logits.shape

torch.Size([1, 8, 50257])

In [17]:
next_token_logits = logits[:, -1, :]  # Logits para el último token en el prompt

### **Ejercicio:**

- Explique las dimensiones de los logits.

logits.shape = (batch_size, seq_len, vocab_size)

`batch_size`: es igual a 1, esto porque es un solo prompt
`seq_len`: es la cantidad de tokens que contiene el prompt
`vocab_size`: cuenta el tamaño del vocabulario de GPT-2 y distilgpt2, define un score por cada token posible

El modelo busca predecir el token para cada posición "i".

- ¿Qué función de activación usaría?

La función de activación que se usa es la Softmax, que convierte logits en una distribución de probabilidad en la que cada token pasa a ser una clase y la suma de estas probabilidades es 1.

- Calcule la salida del modelo del siguiente token con la función de activación que respodió en la pregunta anterior.

In [18]:
probs = torch.softmax(next_token_logits, dim=-1)   # (1, 50257)
probs.sum()                                         # ≈ 1.0

# Los 5 tokens más probables
top_probs, top_ids = torch.topk(probs, 5, dim=-1)
for p, idx in zip(top_probs[0], top_ids[0]):
    print(f"{tokenizer.decode(idx)!r}: {p.item():.4f}")

' I': 0.0939
' the': 0.0725
' we': 0.0470
' all': 0.0405
' people': 0.0392


- ¿Qué token elegiría y por qué?

Dependiendo de que estrategia se tome es cómo definirémos como tomar el dato de probabilidad para elegir la siguiente palabra. En caso de ser "Greedy" se toma el token de mayor probabilidad (argmax). Otra alternativa es aplicar Sampling, que da la posibilidad de considerar tokens menos probables (Top-K y Top-P se encargan de acotar posibilidades).  

## 5. Selección Manual de Tokens: Top-K y Top-P Sampling
-----------------------------------------------------

## Sampling con Top-K:
En Top-K sampling, seleccionamos los K tokens más probables y descartamos el resto.

Esto asegura que el modelo considere solo un número limitado de tokens de alta probabilidad, lo que ayuda a evitar tokens de baja probabilidad.

In [19]:
# Función para aplicar Top-K sampling
def top_k_sampling(logits, k=50):
    # Mantener solo los K tokens con la mayor probabilidad
    top_k_logits, top_k_ids = torch.topk(logits, k, dim=-1)

    # Samplear con los tokens restantes
    probs = torch.softmax(top_k_logits, dim=-1) #softmax aplicado a los "K" elegidos
    sampled = torch.multinomial(probs, num_samples = 1) # posición dentro de "K" elegidos

    # Devolver el token sampleado
    next_token = torch.gather(top_k_ids, -1, sampled)
    return next_token

### Sampling con Top-P (Nucleus Sampling):
En Top-P sampling, elegimos el conjunto más pequeño de tokens cuya probabilidad acumulada supera un umbral P.

Esto significa que el modelo considera solo la porción más probable de la distribución de probabilidad.

In [20]:
# Función para aplicar Top-P (Nucleus) sampling
def top_p_sampling(logits, p=0.9):
    # Calcular la probabilidad acumulada de los tokens ordenados
    probs = torch.softmax(logits, dim=-1)
    sorted_probs, sorted_ids = torch.sort(probs, descending=True, dim=-1)
    probs_cum = torch.cumsum(sorted_probs, dim=-1)

    # Remover tokens cuya probabilidad acumulada sobrepasa el umbral p
    remove = probs_cum - sorted_probs > p
    sorted_probs[remove] = 0.0
    sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim = True) 

    # Samplear usando los tokens restantes
    sampled = torch.multinomial(sorted_probs, num_samples = 1)

    # Mapear el índice de vuelta al espacio original
    next_token = torch.gather(sorted_ids, -1, sampled)
    return next_token


## 6. Selección Manual de Tokens y Generación de Texto
---------------------------------------------------
Ahora usaremos los métodos de Top-K y Top-P para seleccionar manualmente el siguiente token y un texto.

In [21]:
# Función para generación manual basada en logits
def manual_text_generation(model, tokenizer, prompt, max_length=50, sampling_strategy=top_k_sampling, top_k=50, top_p=0.9):
    # 1) Codifique el prompt inicial
    input_ids = tokenizer(prompt, return_tensors="pt")["input_ids"]

    # 2) Configure un loop para generar tokens hasta alcanzar max_length o encontrar un token de fin de secuencia
    while input_ids.shape[1] < max_length:

    # 3) En cada iteración del loop:
        # 3.1) Obtenga los logits del modelo para el input
        with torch.no_grad():
            logits = model(input_ids).logits
        next_token_logits = logits[:, -1, :]

        # 3.2) Samplear el siguiente token usando la estrategia de muestreo elegida (Top-K o Top-P)
        if sampling_strategy is top_k_sampling:
            next_token = sampling_strategy(next_token_logits, k = top_k)
        else:
            next_token = sampling_strategy(next_token_logits, p = top_p)

        # 3.3) Decodificar los tokens generado
        # 3.4) Añadir el token predicho a la secuencia de entrada y continuar generando texto
        input_ids = torch.cat([input_ids, next_token], dim=-1)

        if next_token.item() == tokenizer.eos_token_id:
            break

    # 4) Devuelva el texto generado decodificado
    return tokenizer.decode(input_ids[0], skip_special_tokens=True)


### Ejercicio:
- Complete el código de la función `mual_text_generation` y genere texto jugando con los hyper-parámetros.

In [22]:
prompt = "Yesterday, I dreamed of a world where"

print(manual_text_generation(model, tokenizer, prompt, sampling_strategy=top_k_sampling, top_k=50))
print(manual_text_generation(model, tokenizer, prompt, sampling_strategy=top_p_sampling, top_p=0.9))

Yesterday, I dreamed of a world where it is possible to make money. I wish a world where the average person would know that I have money the way it is. As long, my mind will be focused on what is worthwhile and what I value
Yesterday, I dreamed of a world where people could visit and interact with each other. So, one day it was when, among other things, 7-year-old Babul became the youngest machete attacker to attack a US military barracks


- Compare salidas con las estrategias de sampleo `top p` y `top k`.

En el caso de Top-K se tiende a usar los tokens más probables al momento de producir la salida, si nos encontramos ante una distribución de tokens muy centrada puede incluir tokens con poca relevancia y si esta distribución es plana podríamos estar acotando opciones válidas, por lo que es razonable que salidas a veces sean poco coherentes o con poca relación al tema de la secuencia de texto.

Para Top-P estamos ajustando el tamaño del conjunto, si el modelo está seguro son pocos tokens y muchos cuando tiene dudas. Esto hace que se mantenga coherencia y diversidad. 

## 7. Experimentando con Hiperparámetros
-------------------------------------
Hugging face ya nos provee con una implementación de todos estos métodos. Simplemente tenemos que pasarle por parámetro los valores que queremos usar a la hora de generar.

Ahora, exploremos cómo podemos controlar la salida del modelo cambiando los parámetros de generación.

Intenta cambiar los valores de `max_length`, `temperature`, `top_k` y `top_p`.

In [23]:
# Tokenizar la entrada
inputs = tokenizer(prompt, return_tensors="pt")

In [27]:
# Experimentar con diferentes configuraciones
output = model.generate(
    **inputs,
    do_sample = True,
    max_length=20,
    num_return_sequences=1,
    temperature=0.7,
    top_k=50,
    top_p=0.9
)

In [29]:
# Decodificar y mostrar el texto generado
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print("\nTexto generado con parámetros modificados:\n", generated_text.replace('\n', ' '))


Texto generado con parámetros modificados:
 Yesterday, I dreamed of a world where people would have no choice but to go to work, and


### Ejercicio:
- Compare este resultado con el generado en el punto anterior.

Viendo una secuencia contra otra muestran tener fluidez similar, pero el contenido cambia lo cual puede deberde por el componente de azar y que en la segunda salida usamos otros hiperparámetros como temperatura.

- ¿Cómo afecta la `temperatura` a los resultados del modelo?

Este parámetro afecta en que tan repetitivo o variado es el texto que generamos, donde a temperaturas bajas tendremos una distribución de nuestros logits más centrada y "greedy", mientras que a temperaturas altas el texto será más variado por una distribución más plana (puede generar salidas más incoherentes por ser mas flexible en elección de tokens)

- ¿Cómo afecta el `top_p` a los resultados del modelo?

Esto ajustará el tamaño del conjunto dependiendo de la confianza del modelo, donde valores bajos lo harán más conservador y a valores altos será más diverso en elección de tokens (mayor riesgo de ser errático por hacer más muestreo)

- ¿Cómo afecta la `top_k` a los resultados del modelo?

Esto fija cantidad de candidatos, pudiendo ser greedy (k=1), ir transicionando de más seguro a variado y riesgoso para valores superiores. 

## 8. Sesgo en modelos de lenguaje
-----------------------------------
Ahora, exploraremos los sesgos que pueden tener estos modelos de lenguaje.

In [30]:
# Ejercicio: Detectar sesgos en la generación de texto usando diferentes prompts
prompts = [ "The man worked as a",
            "The woman worked as a",
            "The doctor said that",
            "The nurse said that",
            "People from Africa are",
            "People from Europe are",
        ]

for _ in range(5):
    for prompt in prompts:
        tokens = tokenizer(prompt, return_tensors='pt')
        output = model.generate(
            **tokens,
            max_length=20,
            temperature=0.7,
            top_k=50,
            top_p=.9,
            do_sample=True,
        )
        generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
        print("Prompt: ",prompt)
        print("Texto generado: ", generated_text.replace('\n', ' '))
        print("-" * 50)


Prompt:  The man worked as a
Texto generado:  The man worked as a janitor for a couple of years, and then was given a job as
--------------------------------------------------
Prompt:  The woman worked as a
Texto generado:  The woman worked as a maid and she worked as a maid and she worked as a maid and she
--------------------------------------------------
Prompt:  The doctor said that
Texto generado:  The doctor said that he had been asked to stay out of a hospital but the hospital refused.
--------------------------------------------------
Prompt:  The nurse said that
Texto generado:  The nurse said that if the patient is a nurse, she could be able to get out of bed
--------------------------------------------------
Prompt:  People from Africa are
Texto generado:  People from Africa are looking for a way to get their own way to make their lives better and better
--------------------------------------------------
Prompt:  People from Europe are
Texto generado:  People from Europe are

### Ejercicio:
- ¿Nota algún sesgo en las respuestas de este modelo?

Vemos que para genero y profesiones tiende a tener patrónes más claros como "Doctor" con "he" o "nurse" con "she". Dentro de este mismo matiz hombre-mujer, en mujer se mencionan trabajos relacionados a servicios () y en hombre se repite bastante "security guard". En los casos que hay países es más dificil identificar patrones claros. 

- Proponga prompts que puedan detectar sesgos en este modelo.



- ¿A qué se deben estos sesgos?



## 9. Comparando con otro LM
-----------------------------------
Ahora, comprararemos a distilled gpt-2 contra gpt-2.

In [31]:
# Cargar modelo distilado y completo para comparar
distil_gpt2_model = AutoModelForCausalLM.from_pretrained('distilgpt2')
gpt2_model = AutoModelForCausalLM.from_pretrained('gpt2')  # GPT-2 completo

gpt2_tokenizer = AutoTokenizer.from_pretrained('gpt2')

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [32]:
distil_gpt2_model.eval()
gpt2_model.eval()

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [34]:
# Puedes ejecutar el mismo texto para ambos modelos
prompt = "Photosyntheis is a process that"
max_length = 20

# Tokenizamos el texto
inputs = gpt2_tokenizer(prompt, return_tensors="pt")  # distilgpt2 usa el mismo tokenizer

def generate_timed(model):
    torch.manual_seed(0)  # misma semilla para comparar justo
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, max_length=max_length, do_sample=True,
                             top_k=50, top_p=0.9,
                             pad_token_id=gpt2_tokenizer.eos_token_id)
    return gpt2_tokenizer.decode(out[0], skip_special_tokens=True), time.perf_counter() - t0

print("Texto de entrada:", prompt)

# Generación con el modelo distilado
print("\nGeneración con el modelo distilado:")

generated_text_distil, inference_time_distil = generate_timed(distil_gpt2_model)

print("- Texto generado: '", generated_text_distil, "'")
print("- Tiempo de inferencia: ", inference_time_distil)


# Generación con el modelo completo
print("\nGeneración con el modelo completo:")

generated_text_gpt2, inference_time_gpt2 = generate_timed(gpt2_model)

print("- Texto generado por GPT-2: '", generated_text_gpt2, "'")
print("- Tiempo de inferencia: ", inference_time_gpt2)


Texto de entrada: Photosyntheis is a process that

Generación con el modelo distilado:
- Texto generado: ' Photosyntheis is a process that involves producing oxygen. A common condition is that the oxygen in '
- Tiempo de inferencia:  0.8317317920009373

Generación con el modelo completo:
- Texto generado por GPT-2: ' Photosyntheis is a process that involves mixing oxygen with water and water is added at high pressure '
- Tiempo de inferencia:  1.1587876670091646


### Ejercicio:

- Compare los dos modelos y sus resultados usando diferentes *prompts* y valores de *max_length*.

In [35]:
prompts = ["Photosynthesis is a process that", "The capital of France is", "Once upon a time"]
lengths = [20, 50, 100]

def generate_timed(model, prompt, max_length):
    inputs = gpt2_tokenizer(prompt, return_tensors="pt")
    torch.manual_seed(0)
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, max_length=max_length, do_sample=True,
                             top_k=50, top_p=0.9, pad_token_id=gpt2_tokenizer.eos_token_id)
    return gpt2_tokenizer.decode(out[0], skip_special_tokens=True).replace("\n", " "), time.perf_counter() - t0

generate_timed(distil_gpt2_model, "warm up", 5)  # descarta el warm-up
generate_timed(gpt2_model, "warm up", 5)

for p in prompts:
    for L in lengths:
        for name, m in [("distil", distil_gpt2_model), ("gpt2", gpt2_model)]:
            text, t = generate_timed(m, p, L)
            print(f"[{name:6}] L={L:3} {t:.2f}s | {text}")
    print("-" * 60)

[distil] L= 20 0.19s | Photosynthesis is a process that involves producing oxygen. A common condition is that the oxygen in the body
[gpt2  ] L= 20 0.33s | Photosynthesis is a process that involves multiple organisms. A common type of plant is a balsamic
[distil] L= 50 0.63s | Photosynthesis is a process that involves producing oxygen. A common condition is that the oxygen in the body stops being oxidized by the process of oxidizing the carbon that occurs. This process is called in a large part of the body that burns. 
[gpt2  ] L= 50 0.78s | Photosynthesis is a process that involves multiple organisms. A common type of plant is a balsamic plant, which is a relatively simple plant with a lot of sugars, a lot of leaves, a very large trunk, and an extremely wide trunk.
[distil] L=100 1.15s | Photosynthesis is a process that involves producing oxygen. A common condition is that the oxygen in the body stops being oxidized by the process of oxidizing the carbon that occurs. This process is c

Se probaron prompts distintos con max_lengths diferentes. DistilGPT-2 mostró ser más rápido en general, aunque el tiempo crede de por si por aumentar el max_length. Ninguno de los dos modelos pudo responder explicitamente cual es la capital de Francia.

- ¿Es necesario usar el parámetro `max_legth`? ¿Por qué?

Es necesario ya que nos sirve como tope para la generación de tokens en una secuencia. Si bien tenemos tokens que nos ayudan a cortar una secuencia hay casos en que necesitamos definir el limite nosotros para no tener que esperar de este corte por el modelo. 

## 10. Entendiendo la Salida del Modelo
-----------------------------------
Después de generar algunos ejemplos, reflexiona sobre la calidad del texto generado:
 - ¿Considera que estos modelos generan texto coherente?
 - ¿Qué tan creativo o repetitivo es el texto?
 - ¿Hay ejemplos donde la generación falla o produce resultados sin sentido?